# Creating Piecewise Inequality Bounds

When you only need a one-sided bound from a piecewise curve — `y ≤ f(x)` for a concave upper envelope, `y ≥ f(x)` for a convex lower envelope — `add_piecewise_formulation` accepts an optional sign as the third tuple element:

```python
m.add_piecewise_formulation(
    (power, power_pts, "<="),  # power ≤ f(fuel) — output curtailable
    (fuel,  fuel_pts),         # equality role
)
```

The pay-off is a pure-LP encoding when the curve's curvature matches the sign — no SOS2, no binaries. This notebook covers the geometry of the feasible region, the curvature × sign combinations that unlock the LP path, and what happens when they don't match.

For the formulation math see the [reference page](piecewise-linear-constraints); for the all-equality variant and other features see [Creating Piecewise Linear Constraints](piecewise-linear-constraints-tutorial).

## Tuple roles

| Tuple form | Role | What it constrains |
|---|---|---|
| `(expr, breaks)` | `==` (equality) | With 2+ equality tuples sharing weights, the joint point lies on the curve. With 1 equality + 1 bounded, the equality tuple's marginal feasible set is just its breakpoint domain `[x_min, x_max]` — one coordinate alone can't locate a curve point. |
| `(expr, breaks, "<=")` | bounded above | `expr ≤ f(other tuples)`. |
| `(expr, breaks, ">=")` | bounded below | `expr ≥ f(other tuples)`. |

Currently at most one tuple may carry a non-equality sign, and 3+ tuples must all be equality. Multi-bounded and N≥3 inequality cases aren't supported yet — if you have a concrete use case, please open an issue at https://github.com/PyPSA/linopy/issues so we can scope it properly.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import linopy

## Setup — a concave production curve

A concave, monotonically increasing curve maps fuel input to the maximum power the plant can deliver.  Bounding `power` by this curve with `<=` says the unit may run *at or below* the maximum — power output is curtailable for a given fuel draw.  Concave + `<=` is exactly the combination that lets the LP method apply.

In [ ]:
fuel_pts = np.array([0.0, 36.0, 84.0, 170.0])
power_pts = np.array([0.0, 30.0, 60.0, 100.0])  # slopes 0.83, 0.62, 0.46 (concave)

fig, ax = plt.subplots(figsize=(5, 4))
ax.plot(fuel_pts, power_pts, "o-", color="C0", lw=2)
ax.set(xlabel="fuel", ylabel="power", title="Concave production curve f(fuel)")
ax.grid(alpha=0.3)
plt.tight_layout()

## Three methods, identical feasible region

With `power` bounded `<=` and our concave curve, the three methods give the **same** feasible region for `fuel ∈ [0, 170]`:

- **`method="lp"`** — tangent lines + domain bounds. No auxiliary variables.
- **`method="sos2"`** — lambdas + SOS2 + a split link: equality for the equality-signed tuple, signed for the bounded one. Solver picks the piece.
- **`method="incremental"`** — delta fractions + binaries + split link. Same mathematics, MIP encoding instead of SOS2.

`method="auto"` dispatches to `"lp"` whenever applicable — it's always preferable because it's pure LP.

Let's verify they produce the same solution at `fuel=60`, where `f(60)=45`.

In [ ]:
def solve(method, fuel_val):
    m = linopy.Model()
    fuel = m.add_variables(lower=0, upper=170, name="fuel")
    power = m.add_variables(lower=0, upper=100, name="power")
    m.add_piecewise_formulation(
        (power, power_pts, "<="),  # power ≤ f(fuel) — curtailable output
        (fuel, fuel_pts),  # equality role (domain-bounded to [0, 170])
        method=method,
    )
    m.add_constraints(fuel == fuel_val)
    m.add_objective(-power)  # maximise power output against the upper bound
    m.solve(solver_name="highs", output_flag=False)
    return float(m.solution["power"]), list(m.variables), list(m.constraints)


for method in ["lp", "sos2", "incremental"]:
    power_val, vars_, cons_ = solve(method, 60)
    print(f"{method:12}: power={power_val:.2f}  vars={vars_}  cons={cons_}")

All three give `power=45` at `fuel=60` (which is `f(60)` exactly) — the math is equivalent. The LP method is strictly cheaper: no auxiliary variables, just three chord constraints and two domain bounds.

The SOS2 and incremental methods create lambdas (or deltas + binaries) and split the link into an equality constraint for the equality-signed tuple plus a signed link for the bounded tuple — but the feasible region is the same.

## Visualising the feasible region

The feasible region for `(fuel, power)` with `power` bounded `<=` is the **hypograph** of `f` restricted to the fuel domain:

$$\{ (\mathrm{fuel}, \mathrm{power}) : 0 \le \mathrm{fuel} \le 170,\ \mathrm{power} \le f(\mathrm{fuel}) \}$$

Below we colour feasible points green, infeasible ones red:

- `(60, 30)` — under the curve, `30 ≤ f(60)=45` ✓
- `(60, 45)` — on the curve ✓
- `(60, 55)` — above `f(60)`, infeasible ✗
- `(180, 50)` — fuel beyond domain, infeasible ✗

In [ ]:
def in_hypograph(fx, py):
    if fx < fuel_pts[0] or fx > fuel_pts[-1]:
        return False
    return py <= np.interp(fx, fuel_pts, power_pts)


xx, yy = np.meshgrid(np.linspace(-10, 200, 200), np.linspace(-10, 120, 200))
region = np.vectorize(in_hypograph)(xx, yy)

test_points = [(60, 30), (60, 45), (60, 55), (180, 50)]

fig, ax = plt.subplots(figsize=(6, 5))
ax.contourf(xx, yy, region, levels=[0.5, 1], colors=["lightsteelblue"], alpha=0.5)
ax.plot(fuel_pts, power_pts, "o-", color="C0", lw=2, label="f(fuel)")
for fx, py in test_points:
    feas = in_hypograph(fx, py)
    ax.scatter(
        [fx], [py], color="green" if feas else "red", zorder=5, s=80, edgecolors="black"
    )
    ax.annotate(f"({fx}, {py})", (fx, py), textcoords="offset points", xytext=(8, 5))
ax.set(
    xlabel="fuel",
    ylabel="power",
    title="sign='<=' feasible region — hypograph of f(fuel) on [0, 170]",
)
ax.grid(alpha=0.3)
ax.legend()
plt.tight_layout()

## When is LP the right choice?

`tangent_lines` imposes the **intersection** of chord inequalities. Whether that intersection matches the true hypograph/epigraph of `f` depends on the curvature × sign combination:

| curvature | bounded `<=` | bounded `>=` |
|-----------|--------------|--------------|
| **concave** | **hypograph (exact ✓)** | **wrong region** — requires `y ≥ max_k chord_k(x) > f(x)` |
| **convex**  | **wrong region** — requires `y ≤ min_k chord_k(x) < f(x)` | **epigraph (exact ✓)** |
| linear      | exact | exact |
| mixed (non-convex) | convex hull of `f` (wrong for exact hypograph) | concave hull of `f` (wrong for exact epigraph) |

In the ✗ cases, tangent lines do **not** give a loose relaxation — they give a **strictly wrong feasible region** that rejects points satisfying the true constraint. Example: for a concave `f` with `y ≥ f(x)`, the chord of any piece extrapolated over another piece's x-range lies *above* `f`, so `y ≥ max_k chord_k(x)` forbids `y = f(x)` itself.

`method="auto"` dispatches to LP only in the two **exact** cases (concave + `<=` or convex + `>=`). For the other combinations it falls back to SOS2 or incremental, which encode the hypograph/epigraph exactly via discrete piece selection.

`method="lp"` explicitly forces LP and raises on a mismatched curvature rather than silently producing a wrong feasible region.

For **non-convex** curves with either sign, the only exact option is a piecewise formulation. That's what the bounded-tuple path does internally: it falls back to SOS2/incremental with the sign on the bounded link. No relaxation, no wrong bounds.

In [ ]:
# 1. Non-convex curve: auto falls back (LP relaxation would be loose)
fuel_nc = [0, 50, 100, 170]
power_nc = [0, 60, 30, 100]  # slopes change sign → mixed convexity

m1 = linopy.Model()
fuel1 = m1.add_variables(lower=0, upper=170, name="fuel")
power1 = m1.add_variables(lower=0, upper=100, name="power")
f1 = m1.add_piecewise_formulation((power1, power_nc, "<="), (fuel1, fuel_nc))
print(f"non-convex + '<=' → {f1.method}")

# 2. Concave curve + sign='>=': LP would be loose → auto falls back to MIP
m2 = linopy.Model()
fuel2 = m2.add_variables(lower=0, upper=170, name="fuel")
power2 = m2.add_variables(lower=0, upper=100, name="power")
f2 = m2.add_piecewise_formulation(
    (power2, list(power_pts), ">="), (fuel2, list(fuel_pts))
)
print(f"concave + '>='    → {f2.method}")

# 3. Explicit method="lp" with mismatched curvature raises
try:
    m3 = linopy.Model()
    fuel3 = m3.add_variables(lower=0, upper=170, name="fuel")
    power3 = m3.add_variables(lower=0, upper=100, name="power")
    m3.add_piecewise_formulation(
        (power3, list(power_pts), ">="), (fuel3, list(fuel_pts)), method="lp"
    )
except ValueError as e:
    print(f"lp(concave, '>=')  → raises: {e}")

## Summary

- **One bounded tuple + a 2-variable formulation** gives a hypograph (`<=`) or epigraph (`>=`) feasible region.
- **Curvature × sign matching** — concave + `<=` or convex + `>=` — lets `method="auto"` skip MIP entirely. Mismatched combinations fall back to SOS2/incremental with a signed link.
- **`method="lp"` is strict** — it raises on a mismatched curvature rather than silently encoding the wrong region.
- At most one tuple may carry a non-`==` sign, and 3+ tuples must all be `==`. Multi-bounded / N≥3 inequalities — open an issue at https://github.com/PyPSA/linopy/issues.

**See also**: [reference page](piecewise-linear-constraints) for the formulation math, [Creating Piecewise Linear Constraints](piecewise-linear-constraints-tutorial) for all-equality, unit commitment, CHP, fleets, slopes.